<a href="https://colab.research.google.com/github/sanej/operational-readiness-intelligence/blob/main/operational_readiness_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Industrial Operations Assistant: Mistral-Powered RAG

## Step 1 — Business problem and hypothesis

Pharma, semiconductor, energy, manufacturing, and chemical operations share the same bottleneck: critical decisions depend on fragmented, versioned procedures, inspections, permits, maintenance records, and corrective actions.

**Hypothesis:** A Mistral-based RAG assistant can reduce evidence-review time while improving traceability by retrieving current evidence, citing material claims, and surfacing conflicts or gaps—while qualified personnel retain final authority.

This notebook walks through the requested pipeline in eight inspectable steps: problem → setup → ingest → chunk → embed/index → retrieve → answer → evaluate/scale.

## Step 2 — Environment and secure authentication

The notebook uses the Mistral API, LangChain document utilities, and an in-memory FAISS vector index. In Colab, store the key under **Secrets → MISTRAL_API_KEY**; outside Colab, provide it as an environment variable.

In [1]:
import getpass
import importlib.util
import os
import subprocess
import sys

packages = {
    "mistralai": "mistralai",
    "langchain_mistralai": "langchain-mistralai",
    "langchain_community": "langchain-community",
    "langchain_text_splitters": "langchain-text-splitters",
    "faiss": "faiss-cpu",
    "pypdf": "pypdf",
}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

api_key = os.environ.get("MISTRAL_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("MISTRAL_API_KEY")
    except (ImportError, KeyError, TypeError):
        pass
if not api_key:
    api_key = getpass.getpass("Mistral API key: ")
if not api_key:
    raise ValueError("MISTRAL_API_KEY is required")

os.environ["MISTRAL_API_KEY"] = api_key
print("Environment ready; Mistral API key loaded securely.")

Environment ready; Mistral API key loaded securely.


## Step 3 — Ingest representative operational evidence

The small corpus deliberately mixes a procedure, an incident log, a permit, and two PDF records. This keeps the demo understandable while representing the fragmented evidence found in real operational reviews.

In [2]:
from pathlib import Path
from urllib.request import urlretrieve

documents = [
    {
        "source": "SOP-702-Maintenance",
        "text": "Standard Operating Procedure for Chemical Reactors: All valves must be inspected every 24 hours. Pressure should not exceed 150 PSI.",
        "date": "2023-10-01",
    },
    {
        "source": "Incident-Log-A",
        "text": "On Oct 15th, Reactor 2 showed a pressure spike of 160 PSI. Maintenance was delayed by 12 hours due to permit backlog.",
        "date": "2023-10-15",
    },
    {
        "source": "Permit-System-v2",
        "text": "Hot work permits require signatures from the site manager and the safety officer. Digital logs must be updated in real time.",
        "date": "2024-01-10",
    },
]

pdf_specs = [
    (
        "PTW-2026-0412-work-permit.pdf",
        Path("sample-documents/industrial/PTW-2026-0412-work-permit.pdf"),
        "https://raw.githubusercontent.com/sanej/operational-readiness-intelligence/main/sample-documents/industrial/PTW-2026-0412-work-permit.pdf",
    ),
    (
        "INSP-2026-003-inspection-finding.pdf",
        Path("sample-documents/pharma/INSP-2026-003-inspection-finding.pdf"),
        "https://raw.githubusercontent.com/sanej/operational-readiness-intelligence/main/sample-documents/pharma/INSP-2026-003-inspection-finding.pdf",
    ),
]

pdf_files = []
download_dir = Path("/content/ori-sample-documents") if Path("/content").exists() else Path("/tmp/ori-sample-documents")
download_dir.mkdir(parents=True, exist_ok=True)
for filename, local_path, url in pdf_specs:
    if local_path.exists():
        pdf_files.append(local_path)
    else:
        destination = download_dir / filename
        if not destination.exists():
            urlretrieve(url, destination)
        pdf_files.append(destination)

print(f"Prepared {len(documents)} text records and {len(pdf_files)} PDFs.")

Prepared 3 text records and 2 PDFs.


## Step 4 — Extract and chunk with provenance

`RecursiveCharacterTextSplitter` keeps nearby sentences together. Source, date, and page metadata remain attached so retrieved evidence can be traced back to the originating record.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
processed_chunks = []
metadatas = []

for document in documents:
    for chunk in text_splitter.split_text(document["text"]):
        processed_chunks.append(chunk)
        metadatas.append({"source": document["source"], "date": document["date"], "page": None})

for pdf_path in pdf_files:
    for document in text_splitter.split_documents(PyPDFLoader(str(pdf_path)).load()):
        processed_chunks.append(document.page_content)
        metadatas.append({
            "source": pdf_path.name,
            "date": "2026-04-12",
            "page": document.metadata.get("page"),
        })

print(f"Created {len(processed_chunks)} chunks with source metadata.")

Created 13 chunks with source metadata.


## Step 5 — Embed and index the chunks

Mistral embeddings convert each chunk into a semantic vector. FAISS provides a transparent in-memory index suitable for this interview prototype; a production deployment would add durable, tenant-isolated storage.

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from mistralai.client import Mistral

embeddings = MistralAIEmbeddings(model="mistral-embed", api_key=api_key)
vector_db = FAISS.from_texts(processed_chunks, embeddings, metadatas=metadatas)
client = Mistral(api_key=api_key)

print(f"Indexed {vector_db.index.ntotal} vectors with mistral-embed.")

Indexed 13 vectors with mistral-embed.


## Step 6 — Retrieve evidence for the question

Similarity search returns a bounded evidence set before generation. Printing the sources makes retrieval observable instead of hiding it inside a chain.

In [5]:
query = "What is the pressure limit in SOP-702 and was it violated in Incident-Log-A?"
retrieved_documents = vector_db.similarity_search(query, k=min(6, vector_db.index.ntotal))

print(f"Question: {query}\n")
for rank, document in enumerate(retrieved_documents, start=1):
    source = document.metadata.get("source", "Unknown")
    page = document.metadata.get("page")
    location = f", page {page + 1}" if isinstance(page, int) else ""
    print(f"{rank}. {source}{location}: {document.page_content[:140].strip()}...")

Question: What is the pressure limit in SOP-702 and was it violated in Incident-Log-A?

1. Incident-Log-A: On Oct 15th, Reactor 2 showed a pressure spike of 160 PSI. Maintenance was delayed by 12 hours due to permit backlog....
2. SOP-702-Maintenance: Standard Operating Procedure for Chemical Reactors: All valves must be inspected every 24 hours. Pressure should not exceed 150 PSI....
3. INSP-2026-003-inspection-finding.pdf, page 1: Classification
Major
Affected equipment
GRN-2100, Line 2 Granulation
Status
OPEN
Linked records
DEV-2026-047, CAPA-2026-019, CC-2026-011
3....
4. INSP-2026-003-inspection-finding.pdf, page 1: to end.
1. Finding
During the internal quality systems audit conducted 2026-04-15, the auditor identified that the controlled
document syste...
5. INSP-2026-003-inspection-finding.pdf, page 1: of this record. The finding remains open and the associated CAPA-2026-019 is overdue.
6. Note
This finding does not constitute a determinati...
6. INSP-2026-003-inspection-findin

## Step 7 — Generate a grounded answer with Mistral

The system prompt constrains the model to retrieved evidence, requires source citations, and treats missing evidence as a valid answer. The model proposes the prose; the evidence remains visible for human verification.

In [6]:
SYSTEM_PROMPT = """You are a technical operations evidence assistant.
Use only the retrieved context. Do not invent procedures, limits, dates, or approvals.
If the evidence is incomplete or conflicting, state that explicitly.
Cite every material claim using [Source: document-name].
Return three short sections: Answer, Evidence, and Gaps or conflicts.
Qualified personnel retain final decision authority."""

def ask_industrial_assistant(question, k=6):
    evidence = vector_db.similarity_search(question, k=min(k, vector_db.index.ntotal))
    context = "\n\n".join(
        f"[Source: {item.metadata.get('source', 'Unknown')}]\n{item.page_content}"
        for item in evidence
    )
    response = client.chat.complete(
        model="mistral-medium-3-5",
        temperature=0.1,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Retrieved context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content, evidence

answer, answer_evidence = ask_industrial_assistant(query)
print(answer)

Answer
The pressure limit in SOP-702 is 150 PSI. It was violated in Incident-Log-A, which recorded a spike of 160 PSI.

Evidence
- SOP-702-Maintenance states: "Pressure should not exceed 150 PSI." [Source: SOP-702-Maintenance]
- Incident-Log-A states: "Reactor 2 showed a pressure spike of 160 PSI." [Source: Incident-Log-A]

Gaps or conflicts
No gaps or conflicts in the evidence.


## Step 8 — Evaluate the prototype and explain the scale path

A tiny deterministic smoke test checks whether retrieval found the two records required for the demonstration. Production evaluation would use a reviewer-labelled gold set for retrieval recall, citation validity, unsupported-claim rate, status accuracy, conflicts, latency, and business review time.

In [7]:
retrieved_sources = {document.metadata.get("source") for document in answer_evidence}
checks = {
    "retrieved SOP": "SOP-702-Maintenance" in retrieved_sources,
    "retrieved incident": "Incident-Log-A" in retrieved_sources,
    "answer includes 150 PSI": "150" in answer,
    "answer includes 160 PSI": "160" in answer,
    "answer cites sources": "[Source:" in answer,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

assert all(checks.values()), "The RAG smoke test did not pass every check."
print("\nEnd-to-end RAG smoke test passed.")

PASS — retrieved SOP
PASS — retrieved incident
PASS — answer includes 150 PSI
PASS — answer includes 160 PSI
PASS — answer cites sources

End-to-end RAG smoke test passed.


### Production scale path

- Replace in-memory FAISS with a durable vector service using tenant and corpus isolation.
- Add asynchronous OCR/ingestion, document versioning, retries, and dead-letter handling.
- Combine semantic retrieval with lexical search and reranking for identifiers, dates, and negations.
- Add claim-level citation validation, contradiction checks, prompt/model/index versioning, and replayable audit records.
- Enforce identity, RBAC, retention, and source-system permissions before retrieval.
- Pilot in shadow mode and promote only when evidence quality and reviewer-time gates hold.

**Result:** the notebook demonstrates the interview’s logically sound RAG pipeline; the ORI application demonstrates how the same hypothesis extends toward a customer-facing deployment.